# 📘 Anomaly-Aware AutoML Backend
This notebook demonstrates an implementation of an anomaly-aware AutoML pipeline with dynamic feature engineering.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from pycaret.classification import setup, compare_models, pull, save_model

import warnings
warnings.filterwarnings('ignore')


## Step 1: Load and Preprocess the Dataset

In [ ]:
# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/valayb78/Fraud-Detection/master/creditcard.csv")

# Basic preprocessing
scaler = StandardScaler()
df['scaled_amount'] = scaler.fit_transform(df[['Amount']])
df['scaled_time'] = scaler.fit_transform(df[['Time']])
df = df.drop(['Time', 'Amount'], axis=1)

# Train-test split
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


## Step 2: Anomaly Detection with Isolation Forest

In [ ]:
iso_forest = IsolationForest(n_estimators=100, contamination=0.0017, random_state=42)
X_train['anomaly_score'] = iso_forest.fit_predict(X_train)

# Flag anomalies
X_train['anomaly'] = X_train['anomaly_score'].apply(lambda x: 1 if x == -1 else 0)
print(f"Detected anomalies: {X_train['anomaly'].sum()}")


## Step 3: Dynamic Feature Engineering

In [ ]:
# Add some interaction features
X_train['V1_V2_ratio'] = X_train['V1'] / (X_train['V2'] + 1e-5)
X_test['V1_V2_ratio'] = X_test['V1'] / (X_test['V2'] + 1e-5)


## Step 4: AutoML Training with PyCaret

In [ ]:
train_data = X_train.copy()
train_data['Class'] = y_train.values

s = setup(data=train_data, target='Class', silent=True, session_id=42, preprocess=True, normalize=True)
best_model = compare_models()
results = pull()
results


## Step 5: Evaluate on Test Set

In [ ]:
from pycaret.classification import predict_model

X_test_full = X_test.copy()
X_test_full['Class'] = y_test.values
preds = predict_model(best_model, data=X_test_full)
print(classification_report(preds['Class'], preds['Label']))
print("AUC-ROC:", roc_auc_score(preds['Class'], preds['Score']))


## Step 6: Save the Model

In [ ]:
save_model(best_model, 'anomaly_aware_model')
